In [1]:
import numpy as np
import pandas as pd
df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/train.csv')

Aim: choose a model for this classification task. 

Potential models: RandomForestClassifier, HistGradientBoostingClassifier, XGBClassifier.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn import set_config
set_config(transform_output='pandas')

# pop target labels
y = df.pop('health_condition')

# custom functions for preprocessing
def add_na_count(X):
    X = X.copy()
    X['na_count'] = X.isna().sum(axis=1)
    return X

def add_prod(X):
    X = X.copy()
    X['activity_x_duration'] = (
        X['ordinal_encoding__physical_activity_level']
        * X['numerical__exercise_duration']
    )
    return X

def drop_id(X):
    X = X.copy()
    return X.drop(columns="id")

# separate pipelines for numerical, ordinal, and nominal data
nom_pre = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='other')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

cats = [
    ['non-veg', 'balanced', 'veg'],
    ['low', 'medium', 'high'],
    ['poor', 'average', 'good'],
    ['sedentary', 'moderate', 'active'],
    ['no', 'occasional', 'yes']
]

ord_pre = Pipeline([
    ('encode', OrdinalEncoder(categories=cats, handle_unknown='use_encoded_value',
        unknown_value=np.nan, encoded_missing_value=np.nan)),
    ('impute', SimpleImputer(strategy='median'))
])

num_pre = Pipeline([
    ('impute', SimpleImputer(strategy='median'))
])

ordinal = ['diet_type', 'stress_level', 'sleep_quality',
       'physical_activity_level', 'smoking_alcohol']
numeric = ['sleep_duration', 'heart_rate', 'bmi',
       'calorie_expenditure', 'step_count', 'exercise_duration',
       'water_intake']

# combine preprocessing steps into one Column Transformer
encode_impute = ColumnTransformer([
    ('gender_encoding', nom_pre, ['gender']),
    ('ordinal_encoding', ord_pre, ordinal),
    ('numerical', num_pre, numeric)
], remainder='passthrough')

In [3]:
# define classification model
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder # for compatibility with XGBC
from sklearn.model_selection import cross_val_score, StratifiedKFold

rfc = RandomForestClassifier(random_state=67, n_estimators=20, max_depth=10, class_weight='balanced')
hgbc = HistGradientBoostingClassifier(random_state=67, learning_rate=0.1, max_depth=10, max_iter=200, class_weight='balanced')
xgbc = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    n_estimators=20,
    learning_rate=0.1,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=67,
    n_jobs=-1
)


for model in [rfc, hgbc, xgbc]:
    # create final classification pipeline
    pipe = Pipeline([
        ('drop_id', FunctionTransformer(drop_id, validate=False)),
        ('na_counter', FunctionTransformer(add_na_count, validate=False)),
        ('encode_and_impute', encode_impute),
        ('add_product', FunctionTransformer(add_prod, validate=False)),
        ('classifier', model)
    ])

    # encode for compatibility with XGBC
    y_encoded = LabelEncoder().fit_transform(y)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=67)

    scores = cross_val_score(
        pipe,
        df,
        y_encoded,
        cv=5,
        scoring="balanced_accuracy",
        n_jobs=-1
    )
    
    print(f'Mean score for {model}: {scores.mean()}')

Mean score for RandomForestClassifier(class_weight='balanced', max_depth=10, n_estimators=20,
                       random_state=67): 0.9291295954775691
Mean score for HistGradientBoostingClassifier(class_weight='balanced', max_depth=10,
                               max_iter=200, random_state=67): 0.9368340818765939
Mean score for XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=10, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=Non

HistGradientBoostingClassifier has the highest balanced accuracy score, and is faster than RandomForestClassifier due to the size of the dataset. The model can be finetuned to maximise this statistic.